# CodeAct: cuando la acción es escribir código

**Lección 5 · Clase 5.4** — hasta ahora nuestros agentes actúan por **tool-calling**: un verbo por turno, con el vocabulario que nosotros definimos (`crear_pedido`, `ejecutar_sql`...). Funciona — y tiene un techo. Si la tarea necesita componer operaciones (agrupar, luego calcular una mediana, luego una regresión, luego un gráfico), cada composición es o una herramienta nueva que alguien tiene que escribir, o una cadena larga de turnos.

**CodeAct** (Wang et al., 2024) invierte el planteo: en vez de darle al modelo verbos sueltos, se le da **un intérprete de Python**. El código *es* la acción — y trae gratis composición, loops, estado y todas las librerías. La pregunta obvia es dónde corre ese código sin riesgo, y esa es la integración de hoy: el **sandbox gestionado de Anthropic**, un contenedor aislado (sin internet, con pandas/numpy/matplotlib preinstalados) que se activa declarando una tool en el request. Nosotros no ejecutamos nada.

> No confundir con las *plataformas* de agentes gestionados (donde el proveedor corre el loop completo). Acá el sandbox es **una tool del Messages API**: nuestro código sigue siendo el que conversa con el modelo.

| | |
|---|---|
| **Primer contacto** | Una llamada con `code_execution`: Claude escribe y corre Python solo. |
| **Datos de verdad** | Subir un CSV al sandbox por la Files API. |
| **La comparación** | El mismo análisis para un agente de tool-calling puro (gpt-5-mini + 2 herramientas estrechas) y para Claude con sandbox — con la respuesta correcta como juez. |
| **Artefactos** | El gráfico se genera en el sandbox y se descarga por la Files API. |
| **Estado** | El contenedor persiste entre llamadas: análisis conversacional. |


In [ ]:
# Esta lección usa el entorno uv del README. Si la corres en Colab, descomenta:
# %pip install -q anthropic==0.121.0 langchain==1.3.14 langchain-core==1.5.3 \
#   langchain-openai==1.4.2 pandas==2.3.3 matplotlib==3.11.1 python-dotenv==1.2.2
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # el .env de la lección gana sobre variables heredadas del entorno

try:
    from google.colab import userdata  # type: ignore
    for llave in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[llave] = userdata.get(llave) or os.environ.get(llave, "")
        except Exception:
            pass
except Exception:
    pass

HAY_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAY_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print("ANTHROPIC_API_KEY presente:", HAY_ANTHROPIC)
print("OPENAI_API_KEY presente:  ", HAY_OPENAI)
if not HAY_ANTHROPIC:
    print("⚠️ Sin ANTHROPIC_API_KEY las secciones del sandbox se saltan (es la única")
    print("   lección de la clase que la usa: console.anthropic.com).")
if not HAY_OPENAI:
    print("⚠️ Sin OPENAI_API_KEY se salta el agente de comparación.")

import urllib.request
from pathlib import Path

BASE_RAW = (
    "https://raw.githubusercontent.com/josepenam/clases-diplomado-gen-ia/main/"
    "class_5_4_integraciones/leccion5_codeact_ejecucion_de_codigo/"
)


def asegurar(nombre: str) -> Path:
    ruta = Path(nombre)
    if not ruta.exists():
        ruta.parent.mkdir(parents=True, exist_ok=True)
        pedido = urllib.request.Request(
            BASE_RAW + nombre, headers={"User-Agent": "Mozilla/5.0 (clase-diplomado-gen-ia)"}
        )
        ruta.write_bytes(urllib.request.urlopen(pedido).read())
    return ruta


RUTA_CSV = asegurar("data/ventas_diarias.csv")
Path("outputs").mkdir(exist_ok=True)

# El modelo del sandbox. claude-haiku-4-5 es la opción económica para practicar.
MODELO_CLAUDE = "claude-opus-5"
MODELO_OPENAI = "gpt-5-mini"


## Primer contacto con el sandbox

Declarar la tool `code_execution` es todo lo que hace falta: es una tool **de servidor** — Anthropic la ejecuta; no hay loop de tool-calling de nuestro lado. La respuesta llega intercalada: texto del modelo, el código que decidió correr (`server_tool_use`) y los resultados de ejecución (`bash_code_execution_tool_result`), con stdout y todo.

El helper de abajo es el corazón de la lección: manda la pregunta, recorre los bloques y maneja los dos detalles operativos del mundo real — `pause_turn` (el turno del servidor se pausó: se reenvía y continúa) y `refusal`.

In [ ]:
import anthropic

TOOL_EJECUCION = {"type": "code_execution_20260120", "name": "code_execution"}

cliente_claude = anthropic.Anthropic() if HAY_ANTHROPIC else None


def preguntar_sandbox(pregunta, container=None, file_id=None, max_reenvios=5, verboso=True):
    """Una conversación de un turno con Claude + sandbox de código.

    Devuelve un dict con: texto (la respuesta), stdouts, codigos (lo que corrió),
    file_ids (archivos generados en el sandbox) y container (para reutilizar).
    """
    contenido = [{"type": "text", "text": pregunta}]
    if file_id:
        contenido.append({"type": "container_upload", "file_id": file_id})
    mensajes = [{"role": "user", "content": contenido}]

    parametros = dict(model=MODELO_CLAUDE, max_tokens=8000,
                      tools=[TOOL_EJECUCION], messages=mensajes,
                      extra_headers={"anthropic-beta": "files-api-2025-04-14"})
    if container:
        parametros["container"] = container

    respuesta = cliente_claude.messages.create(**parametros)
    for _ in range(max_reenvios):
        if respuesta.stop_reason != "pause_turn":
            break
        # el loop del servidor se pausó: se reenvía la conversación y continúa solo
        mensajes = mensajes + [{"role": "assistant", "content": respuesta.content}]
        parametros["messages"] = mensajes
        parametros["container"] = respuesta.container.id
        respuesta = cliente_claude.messages.create(**parametros)

    if respuesta.stop_reason == "refusal":
        return {"texto": "(el modelo declinó la solicitud)", "stdouts": [], "codigos": [],
                "file_ids": [], "container": None}

    resultado = {"texto": "", "stdouts": [], "codigos": [], "file_ids": [],
                 "container": respuesta.container.id if respuesta.container else None}
    for bloque in respuesta.content:
        if bloque.type == "text":
            resultado["texto"] += bloque.text
        elif bloque.type == "server_tool_use":
            resultado["codigos"].append(bloque.input.get("code") or str(bloque.input))
        elif bloque.type == "bash_code_execution_tool_result":
            contenido_bloque = bloque.content
            if getattr(contenido_bloque, "stdout", None):
                resultado["stdouts"].append(contenido_bloque.stdout)
            for ref in getattr(contenido_bloque, "content", None) or []:
                if getattr(ref, "type", "") == "bash_code_execution_output":
                    resultado["file_ids"].append(ref.file_id)

    if verboso:
        print(f"[{len(resultado['codigos'])} ejecuciones en el sandbox]")
        for salida in resultado["stdouts"]:
            print("· stdout:", salida.strip()[:300])
        print()
        print(resultado["texto"])
    return resultado


if HAY_ANTHROPIC:
    primer_contacto = preguntar_sandbox(
        "Calcula la desviación estándar (muestral) de [12, 15, 9, 22, 18, 31, 25, 14] "
        "y verifica el resultado a mano con la fórmula. Responde en español, breve."
    )
else:
    print("⛔ Falta ANTHROPIC_API_KEY.")

Fíjate en lo que **no** hicimos: no escribimos la fórmula, no definimos una herramienta `desviacion_estandar`, no ejecutamos nada en esta máquina. El modelo decidió qué código correr y el sandbox se lo ejecutó.

## Datos de verdad: subir un CSV

El sandbox no tiene internet (a propósito). Los datos entran por la **Files API**: se sube el archivo una vez y se referencia con un bloque `container_upload` — aparece en el directorio de trabajo del contenedor. Nuestro dataset: las ventas diarias por canal de la temporada, con la nieve caída cada día ([`generar_datos.py`](generar_datos.py), semilla fija).

In [ ]:
if HAY_ANTHROPIC:
    subido = cliente_claude.beta.files.upload(
        file=("ventas_diarias.csv", open(RUTA_CSV, "rb"), "text/csv"),
    )
    print("file_id:", subido.id)

    exploracion = preguntar_sandbox(
        "Explora el CSV que te subí: columnas, tipos, cuántas filas, rango de fechas "
        "y qué canales de venta hay. Responde en español, compacto.",
        file_id=subido.id,
    )

## La comparación: tool-calling puro vs code-acting

La tarea es un análisis de verdad, de los que pide un gerente un lunes:

> *"Con las ventas de la temporada: (1) la mediana de ventas diarias por canal, (2) la correlación entre nieve caída y ventas totales del día, y (3) una regresión lineal ventas ~ nieve con su R²."*

Primero la respuesta correcta con pandas, como juez imparcial. Después, los dos competidores:

- El **agente de tool-calling puro** (gpt-5-mini) recibe dos herramientas estrechas, del estilo que definimos en las lecciones anteriores: ver una muestra y calcular agregaciones con un menú fijo (`suma`, `promedio`, `max`, `min`, `contar`). Es el diseño honesto de lo que es tool-calling: el vocabulario que el desarrollador previó.
- **Claude con sandbox** recibe... el CSV. Nada más.

In [ ]:
import pandas as pd

datos = pd.read_csv(RUTA_CSV)

mediana_por_canal = datos.groupby("canal")["ventas_clp"].median()
por_dia = datos.groupby("fecha").agg(ventas=("ventas_clp", "sum"), nieve=("nieve_cm", "first"))
correlacion = por_dia["nieve"].corr(por_dia["ventas"])

import numpy as np

pendiente, intercepto = np.polyfit(por_dia["nieve"], por_dia["ventas"], 1)
r2 = correlacion**2

print("── la respuesta correcta (el juez) ──")
print(mediana_por_canal.to_string())
print(f"correlación nieve↔ventas: {correlacion:.3f}")
print(f"regresión: ventas = {pendiente:,.0f}·nieve + {intercepto:,.0f}   R² = {r2:.3f}")

TAREA_ANALISIS = (
    "Analiza las ventas de la temporada del CSV ventas_diarias.csv "
    "(columnas: fecha, canal, trx, ventas_clp, nieve_cm — la nieve es la del día, "
    "repetida en las filas de cada canal). Necesito: "
    "(1) la MEDIANA de ventas_clp diarias por canal, "
    "(2) la correlación entre nieve caída y ventas totales del día, "
    "(3) una regresión lineal de ventas totales diarias sobre nieve, con su R². "
    "Reporta los números con 3 decimales donde aplique, en español."
)

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


@tool
def ver_muestra_csv(filas: int = 5) -> str:
    """Devuelve las primeras filas del CSV de ventas y los tipos de columna."""
    return datos.head(filas).to_string() + "\n\ntipos:\n" + datos.dtypes.to_string()


@tool
def calcular(columna: str, operacion: str, agrupar_por: str = "") -> str:
    """Calcula una agregación sobre una columna del CSV de ventas.

    operacion: una de 'suma', 'promedio', 'max', 'min', 'contar'.
    agrupar_por: opcional, nombre de columna para agrupar (ej. 'canal').
    """
    operaciones = {"suma": "sum", "promedio": "mean", "max": "max", "min": "min", "contar": "count"}
    if operacion not in operaciones:
        return f"ERROR: operación '{operacion}' no disponible. Menú: {list(operaciones)}"
    if columna not in datos.columns:
        return f"ERROR: columna '{columna}' no existe. Columnas: {list(datos.columns)}"
    if agrupar_por:
        if agrupar_por not in datos.columns:
            return f"ERROR: columna '{agrupar_por}' no existe."
        return datos.groupby(agrupar_por)[columna].agg(operaciones[operacion]).to_string()
    return str(datos[columna].agg(operaciones[operacion]))


if HAY_OPENAI:
    agente_tools = create_agent(
        ChatOpenAI(model=MODELO_OPENAI),
        tools=[ver_muestra_csv, calcular],
        system_prompt=(
            "Eres un analista de datos. Trabajas SOLO con las herramientas disponibles "
            "sobre el CSV de ventas; no inventes números. Si una herramienta no alcanza "
            "para calcular algo exactamente, dilo honestamente. Responde en español."
        ),
    )

    resultado_tools = agente_tools.invoke(
        {"messages": [{"role": "user", "content": TAREA_ANALISIS}]},
        config={"recursion_limit": 40},
    )
    respuesta_tools = resultado_tools["messages"][-1].content
    llamadas_tools = sum(1 for m in resultado_tools["messages"] if type(m).__name__ == "ToolMessage")
    print(f"═══ Agente tool-calling puro ({llamadas_tools} llamadas a herramientas) ═══\n")
    print(respuesta_tools)
else:
    print("⛔ Falta OPENAI_API_KEY.")

In [ ]:
if HAY_ANTHROPIC:
    resultado_sandbox = preguntar_sandbox(TAREA_ANALISIS, file_id=subido.id, verboso=False)
    print(f"═══ Claude + sandbox ({len(resultado_sandbox['codigos'])} ejecuciones de código) ═══\n")
    print(resultado_sandbox["texto"])

### El veredicto

Contrastamos cada respuesta contra el juez. Buscamos tres cosas: la mediana de un canal concreto, la correlación y el R² (con tolerancia, porque hay más de una forma legítima de redondear):

In [ ]:
import re


def numeros_de(texto: str) -> set[float]:
    encontrados = set()
    for pedazo in re.findall(r"-?\d[\d.,]*\d|-?\d", texto or ""):
        for interpretacion in (pedazo.replace(",", ""), pedazo.replace(".", "").replace(",", ".")):
            try:
                encontrados.add(round(float(interpretacion), 3))
            except ValueError:
                pass
    return encontrados


def aparece(valor: float, numeros: set[float], tolerancia: float = 0.02) -> str:
    return "✓" if any(abs(n - valor) <= abs(valor) * tolerancia + 0.02 for n in numeros) else "✗"


OBJETIVOS = {
    "mediana canal web": float(mediana_por_canal["web"]),
    "correlación": round(float(correlacion), 3),
    "R²": round(float(r2), 3),
}

filas_veredicto = []
if HAY_OPENAI:
    numeros = numeros_de(respuesta_tools)
    filas_veredicto.append({"agente": "tool-calling puro",
                            **{k: aparece(v, numeros) for k, v in OBJETIVOS.items()}})
if HAY_ANTHROPIC:
    numeros = numeros_de(resultado_sandbox["texto"] + " " + " ".join(resultado_sandbox["stdouts"]))
    filas_veredicto.append({"agente": "claude + sandbox",
                            **{k: aparece(v, numeros) for k, v in OBJETIVOS.items()}})

if filas_veredicto:
    display(pd.DataFrame(filas_veredicto).set_index("agente"))
    print("(el juez:", ", ".join(f"{k} = {v:,}" for k, v in OBJETIVOS.items()), ")")

El agente de herramientas estrechas hace lo que puede: con `promedio` en el menú aproxima, o declara honestamente que no puede calcular una mediana ni una regresión (nuestro prompt se lo pide — un agente menos honesto habría inventado los números). No es un agente mal hecho: es el techo del vocabulario que le dimos. La respuesta de ingeniería clásica sería "agreguemos una herramienta `mediana`, y una `correlacion`, y una `regresion`..." — y ese pasillo no tiene fin.

El agente que escribe código no tiene ese techo: pandas ya sabía hacer todo.

## Recuperar artefactos: el gráfico

Lo que se genera en el sandbox no muere ahí. Los archivos creados (un PNG de matplotlib, por ejemplo) llegan como referencias con `file_id`, y se descargan por la misma Files API:

In [ ]:
if HAY_ANTHROPIC:
    grafico = preguntar_sandbox(
        "Con el CSV: haz un scatter de nieve_cm vs ventas totales del día, con la recta "
        "de regresión encima, título y ejes en español, y guárdalo como grafico_nieve_ventas.png",
        file_id=subido.id,
        verboso=False,
    )
    print(grafico["texto"][:300])
    print("\narchivos generados en el sandbox:", grafico["file_ids"])

    if grafico["file_ids"]:
        contenido_png = cliente_claude.beta.files.download(grafico["file_ids"][-1])
        ruta_png = Path("outputs/grafico_nieve_ventas.png")
        contenido_png.write_to_file(ruta_png)
        from IPython.display import Image, display as mostrar

        mostrar(Image(str(ruta_png), width=640))

## Estado entre llamadas: el contenedor persiste

Cada respuesta trae un `container.id`. Si lo pasamos en la siguiente llamada, **el mismo contenedor** sigue vivo: los archivos, las variables cargadas, lo instalado. Eso convierte el sandbox en un análisis conversacional — la pregunta de seguimiento no vuelve a subir ni a leer nada:

In [ ]:
if HAY_ANTHROPIC and grafico["container"]:
    seguimiento = preguntar_sandbox(
        "Con los mismos datos que ya tienes cargados: ¿cuál fue el mejor día de la "
        "temporada en ventas totales y cuánto se vendió? ¿Nevó ese día?",
        container=grafico["container"],
    )

## Cuándo cada patrón

| | Tool-calling puro | CodeAct (sandbox) |
|---|---|---|
| **Acciones** | El vocabulario que definiste | Todo lo que Python pueda expresar |
| **Control** | Máximo: cada verbo es auditable y acotable | El código hay que *contenerlo* (sandbox) |
| **Ideal para** | Acciones con efectos (pagar, reservar, escribir en sistemas) | Análisis, transformación, cómputo abierto |
| **Riesgo típico** | El techo del vocabulario | Ejecución de código no confiable |
| **Costo** | Un turno por verbo | Sandbox por hora (gratis hasta 1.550 h/mes por organización, luego ~USD 0,05/h) |

Los dos conviven en un mismo agente: verbos estrechos para los *efectos* (la lección 4 no querría un `exec()` con acceso a la base) y un sandbox para el *cómputo*. La regla de diseño: **el código del modelo nunca corre donde corre tu aplicación** — por eso el sandbox gestionado, y no un `exec()` local, es la integración correcta.

Para seguir tirando del hilo: el paper original de CodeAct ([Wang et al., 2024](https://arxiv.org/abs/2402.01030)); la [documentación de la code execution tool](https://platform.claude.com/docs/en/agents-and-tools/tool-use/code-execution-tool); y el siguiente escalón de esta integración — plataformas donde el proveedor también corre el loop del agente (Managed Agents de Anthropic), que ya es tema de arquitectura de producción.